In [ ]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

In [3]:
df_marking["alcohol"].value_counts()

alcohol
да                78
нет               19
['нет', 'нет']     2
['да', 'да']       1
Name: count, dtype: int64

In [ ]:
import re
import pandas as pd

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

alcohol_keywords = {
    'да': [
        r"в состоянии\s+алкогольного\s+опьянения",
        r"алкогольное\s+опьянение",
        r"находясь\s+в\s+состоянии\s+опьянения",
        r"находился\s+в\s+состоянии\s+алкогольного\s+опьянения",
        r"выпив\s+(алкоголь|спиртное)",
        r"употребив\s+(алкоголь|спиртное)",
        r"был\s+пьян", r"была\s+пьяна",
        r"выпивал\s+(водку|пиво|спиртное)",
        r"влиянием\s+алкоголя",
        r"опьянение,\s+вызванное\s+употреблением\s+алкоголя"
    ],
    'нет': []  # если не найдено ни одного шаблона, то "нет"
}

train_data_alcohol = []
s = 0

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    true_label = str(row.get("alcohol")).strip().lower()
    if true_label not in alcohol_keywords:
        continue

    found = False
    for pattern in alcohol_keywords['да']:  # проверяем только по шаблонам "да"
        match = re.search(pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_alcohol.append((text, {"entities": [(start, end, "ALCOHOL")]}))
            found = True
            break

    if not found:
        train_data_alcohol.append((text, {"entities": []}))
        print(f"Не найдены ключевые слова 'да' в id={idx}")
        s += 1

print(f"Не найдены ключевые слова в {s} примерах")
print(f"TRAIN_DATA_ALCOHOL готово: {len(train_data_alcohol)} примеров")

Не найдены ключевые слова 'да' в id=5
Не найдены ключевые слова 'да' в id=14
Не найдены ключевые слова 'да' в id=19
Не найдены ключевые слова 'да' в id=25
Не найдены ключевые слова 'да' в id=32
Не найдены ключевые слова 'да' в id=35
Не найдены ключевые слова 'да' в id=54
Не найдены ключевые слова 'да' в id=60
Не найдены ключевые слова 'да' в id=63
Не найдены ключевые слова 'да' в id=64
Не найдены ключевые слова 'да' в id=66
Не найдены ключевые слова 'да' в id=88
Не найдены ключевые слова 'да' в id=96
Не найдены ключевые слова в 13 примерах
TRAIN_DATA_ALCOHOL готово: 97 примеров


In [8]:
with open("description.txt", "w", encoding="utf-8") as f:
    f.write(df_marking.at[5, "description"])

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("ALCOHOL")

examples = []
for text, annot in train_data_alcohol:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_alcohol_model")
print("Модель сохранена в 'ner_alcohol_model'")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch 1, Losses: {'ner': 265998.57185895473}
Epoch 2, Losses: {'ner': 468.0312253103202}
Epoch 3, Losses: {'ner': 314.54531082934733}
Epoch 4, Losses: {'ner': 6111.358701088385}
Epoch 5, Losses: {'ner': 205.68623983559038}
Epoch 6, Losses: {'ner': 158.20626860238835}
Epoch 7, Losses: {'ner': 135.15170481928138}
Epoch 8, Losses: {'ner': 97.18889115514473}
Epoch 9, Losses: {'ner': 84.14914896036328}
Epoch 10, Losses: {'ner': 103.8293962653256}
Epoch 11, Losses: {'ner': 94.07702753900085}
Epoch 12, Losses: {'ner': 91.71440954631636}
Epoch 13, Losses: {'ner': 78.08928369941844}
Epoch 14, Losses: {'ner': 68.90417369758694}
Epoch 15, Losses: {'ner': 73.36827988123001}
✅ Модель сохранена в 'ner_alcohol_model'


In [ ]:
import spacy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nlp = spacy.load("ner_alcohol_model")

preds = []
true_labels = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    true_label = str(row.get("alcohol")).strip().lower()

    if true_label not in ["да", "нет"]:
        continue

    doc = nlp(text)
    predicted_label = "да" if any(ent.label_ == "ALCOHOL" for ent in doc.ents) else "нет"

    preds.append(predicted_label)
    true_labels.append(true_label)

print("\nМетрики качества для признака 'alcohol':")
print(f"Accuracy:  {accuracy_score(true_labels, preds):.2f}")
print(f"Precision: {precision_score(true_labels, preds, pos_label='да'):.2f}")
print(f"Recall:    {recall_score(true_labels, preds, pos_label='да'):.2f}")
print(f"F1-score:  {f1_score(true_labels, preds, pos_label='да'):.2f}")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Метрики качества для признака 'alcohol':
Accuracy:  0.71
Precision: 0.90
Recall:    0.72
F1-score:  0.80


Обучаем на 80, тестируем на 20.

In [ ]:
from sklearn.model_selection import train_test_split
from spacy.training import Example
import random
import spacy
import warnings

train_examples, valid_examples = train_test_split(train_data_alcohol, test_size=0.2, random_state=42)

nlp = spacy.blank("ru")

ner = nlp.add_pipe("ner")
ner.add_label("ALCOHOL")

1

In [ ]:
train_data = [Example.from_dict(nlp.make_doc(text), annot) for text, annot in train_examples]

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(train_data)
    losses = {}
    batches = spacy.util.minibatch(train_data, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

Epoch 1, Losses: {'ner': 258042.90863239765}
Epoch 2, Losses: {'ner': 322.4002027653265}
Epoch 3, Losses: {'ner': 150.81057113506563}
Epoch 4, Losses: {'ner': 128.0934209118646}
Epoch 5, Losses: {'ner': 1234.3926105306236}
Epoch 6, Losses: {'ner': 241.39399639199928}
Epoch 7, Losses: {'ner': 130.64428142911729}
Epoch 8, Losses: {'ner': 100.59022177887923}
Epoch 9, Losses: {'ner': 73.01766684601021}
Epoch 10, Losses: {'ner': 62.49095852793079}
Epoch 11, Losses: {'ner': 64.73844410023396}
Epoch 12, Losses: {'ner': 55.827487067691195}
Epoch 13, Losses: {'ner': 59.066094783483145}
Epoch 14, Losses: {'ner': 51.68025669047541}
Epoch 15, Losses: {'ner': 52.80065622638193}


In [ ]:
nlp.to_disk("ner_s_alcohol_model")
print("Модель сохранена в 'ner_s_alcohol_model'")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = []
y_pred = []

for text, annot in valid_examples:
    doc = nlp(text)
    pred = any(ent.label_ == "ALCOHOL" for ent in doc.ents)
    true = len(annot["entities"]) > 0

    y_pred.append("да" if pred else "нет")
    y_true.append("да" if true else "нет")

print("Метрики на валидации:")
print("Precision:", precision_score(y_true, y_pred, pos_label="да"))
print("Recall:   ", recall_score(y_true, y_pred, pos_label="да"))
print("F1-score: ", f1_score(y_true, y_pred, pos_label="да"))

🔍 Метрики на валидации:
Precision: 1.0
Recall:    0.7647058823529411
F1-score:  0.8666666666666667
